# 3. Concepts

---

### 3.1 Graph

LangGraph models agent workflows as graphs using three key components:
1. `State`: A shared data structure that represents the current snapshot of your application. It can be any data type, but is typically defined using a shared state schema.

2. `Nodes`: Functions that encode the logic of your agents. They receive the current state as input, perform some computation or side-effect, and return an updated state.

3. `Edges`: Functions that determine which Node to execute next based on the current state. They can be conditional branches or fixed transitions.

The `StateGraph` class is the main graph class to use. This is parameterized by a user defined `State` object.

In [ ]:
# graph_builder = StateGraph(State)

To build your graph, you first define the state, you then add nodes and edges, and then you compile it.

In [ ]:
# graph = graph_builder.compile(...)

### 3.2 State



The first thing you do when you define a graph is define the `State` of the graph. The `State` consists of the ***schema of the graph*** as well as ***reducer*** functions which specify how to apply updates to the state. The schema of the `State` will be the input schema to all `Nodes` and `Edges` in the graph. All `Nodes` will emit updates to the `State` which are then applied using the specified reducer function.

#### Schema

By default, the graph will have the same input and output schemas.

- The main documented way to specify the schema of a graph is by using a `TypedDict`.

- If you want to provide **default values** in your state, use a `dataclass`. 

In [5]:
from typing_extensions import TypedDict
from dataclasses import dataclass

class State1(TypedDict):
    data: int

@dataclass
class State2():
    data: int = 6

state1 = State1()
print(state1)

state2 = State2()
print(state2)

{}
State2(data=6)


#### Multiple schemas

Typically, all graph nodes communicate with a single schema. This means that they will read and write to the same state channels. But, there are cases where we want more control over this:
- Internal nodes can pass information that is not required in the graph’s input / output.
  
- We may also want to use different input / output schemas for the graph. The output might, for example, only contain a single relevant output key.

In [2]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

class InputState(TypedDict):
    user_input: str

class OutputState(TypedDict):
    graph_output: str

class OverallState(TypedDict):
    foo: str
    user_input: str
    graph_output: str

class PrivateState(TypedDict):
    bar: str

def node_1(state: InputState) -> OverallState:
    return {"foo": state["user_input"] + " name"}

def node_2(state: OverallState) -> PrivateState:
    return {"bar": state["foo"] + " is"}

def node_3(state: PrivateState) -> OutputState:
    return {"graph_output": state["bar"] + " Lance"}

builder = StateGraph(OverallState,input_schema=InputState,output_schema=OutputState)
builder.add_node(node_1)
builder.add_node(node_2)
builder.add_node(node_3)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
graph.invoke({"user_input": "My"})

{'graph_output': 'My name is Lance'}

### 3.3 Reducers

Reducers are key to understanding how updates from nodes are applied to the `State`. Each key in the `State` has its own independent reducer function. If no reducer function is explicitly specified then it is assumed that all updates to that key should ***override*** it.

#### Default reducer

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    foo: int
    bar: list[str]

Let’s assume the input to the graph is: `{"foo": 1, "bar": ["hi"]}`.

Let’s then assume the first Node returns `{"foo": 2}`. 

After applying this update, the `State` would then be `{"foo": 2, "bar": ["hi"]}`. 

If the second node returns `{"bar": ["bye"]}` then the `State` would then be `{"foo": 2, "bar": ["bye"]}`


#### Add reducer

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: int
    bar: Annotated[list[str], add]

In this example, we’ve used the `Annotated` type to specify a reducer function (`operator.add`) for the second key (`bar`).

Let’s assume the input to the graph is `{"foo": 1, "bar": ["hi"]}`. 

Let’s then assume the first `Node` returns `{"foo": 2}`.

After applying this update, the State would then be `{"foo": 2, "bar": ["hi"]}`. 

If the second node returns `{"bar": ["bye"]}` then the State would then be `{"foo": 2, "bar": ["hi", "bye"]}`.

#### Overwrite

In some cases, you may want to bypass a reducer and directly overwrite a state value. LangGraph provides the `Overwrite` type for this purpose.

When a node returns a value wrapped with `Overwrite`, the reducer is bypassed and the channel is set directly to that value.

In [1]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import Annotated, TypedDict
from langgraph.types import Overwrite
import operator

class State(TypedDict):
    messages: Annotated[list, operator.add]

def add_message(state: State):
    return {"messages": ["first message"]}

def replace_messages(state: State):
    return {"messages": Overwrite(["replacement message"])}

builder = StateGraph(State)
builder.add_node(add_message)
builder.add_node(replace_messages)
builder.add_edge(START, "add_message")
builder.add_edge("add_message", "replace_messages")
builder.add_edge("replace_messages", END)

graph = builder.compile()

result = graph.invoke({"messages": ["initial"]})
print(result["messages"])

['replacement message']


When nodes execute in parallel, only one node can use `Overwrite` on the same state key in a given super-step. If multiple nodes attempt to overwrite the same key in the same super-step, an `InvalidUpdateError` will be raised.

#### `add_messages`

In some cases, you might want to manually update **existing** messages in your graph state (e.g. human-in-the-loop). If you were to use `operator.add`, the manual state updates you send to the graph would be appended to the existing list of messages, instead of updating existing messages. 

To avoid that, you need a reducer that can keep track of message IDs and overwrite existing messages, if updated. To achieve this, you can use the prebuilt `add_messages` function. For brand new messages, it will simply append to existing list, but it will also handle the updates for existing messages correctly.

In addition to keeping track of message IDs, the `add_messages` function will also automatically serialize/deserialize the format of messages:
- JSON format

- LangChain `Messages` format

In [ ]:
# JSON format
# {"messages": [{"type": "human", "content": "abcde..."}]}

# Langchain Messages format
# {"messages": [HumanMessage(content="abcde...")]}

#### `MessagesState`


Since having a list of messages in your state is so common, there exists a prebuilt state called `MessagesState` which makes it easy to use messages. `MessagesState` is defined with a single messages key which is a list of `AnyMessage` objects and uses the `add_messages` reducer. Typically, there is more state to track than just messages, so we see people subclass this state and add more fields, like:

In [ ]:
from langgraph.graph import MessagesState

class State(MessagesState):
    documents: list[str]

### 3.4 Nodes

In LangGraph, nodes are Python functions that accept the following arguments:
- ***state***: The state of the graph
  
- ***config***: A RunnableConfig object that contains configuration information and tracing information
  
- ***runtime***: A Runtime object that contains runtime context and other information like store and stream_writer

In [ ]:
from typing_extensions import TypedDict
from dataclasses import dataclass
from langgraph.graph import StateGraph

class State(TypedDict):
    user: str
    results: str

@dataclass
class Context:
    user_id: str

builder = StateGraph(State)

def plain_node(state):
    return state

def node_with_config(state, config):
    print("In node:", config["configurable"]["thread_id"])
    return {"results": f"Hello {state['user']}!"}

def node_with_runtime(state, runtime):
    print("In node:", runtime.context.user_id)
    return {"results": f"Hello {state['user']}!"}

builder.add_node(plain_node)
builder.add_node(node_with_config)
builder.add_node(node_with_runtime)

#### Node caching

Specify a cache when compiling a graph (or specifying an entrypoint)

Specify a cache policy for nodes. Each cache policy supports:
- `key_func` used to generate a cache key based on the input to a node, which defaults to a hash of the input with pickle.
  
- `ttl`, the time to live for the cache in seconds. If not specified, the cache will never expire.

In [5]:
import time
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.cache.memory import InMemoryCache
from langgraph.types import CachePolicy

class State(TypedDict):
    x: int
    y: int

builder = StateGraph(State)

def func(state):
    time.sleep(2)
    return {"y": state["x"] + 6}

builder.add_node(func, cache_policy=CachePolicy(ttl=3))
builder.add_edge(START, "func")
builder.add_edge("func", END)

graph = builder.compile(cache=InMemoryCache())
print(graph.invoke({"x": 6}, stream_mode='updates'))
print(graph.invoke({"x": 6}, stream_mode='updates'))

[{'func': {'y': 12}}]
[{'func': {'y': 12}, '__metadata__': {'cached': True}}]


### 3.5 Edges

Edges define how the logic is routed and how the graph decides to stop. This is a big part of how your agents work and how different nodes communicate with each other. There are a few key types of edges:

- Normal Edges: Go directly from one node to the next.
  
- Conditional Edges: Call a function to determine which node(s) to go to next.

- Entry Point: Which node to call first when user input arrives.
- Conditional Entry Point: Call a function to determine which node(s) to call first when user input arrives.

A node can have multiple outgoing edges. All of those destination nodes will be executed in parallel as a part of the next superstep.